# AVAX LONG — Mean reversion ở dải Bollinger dưới, train full grid trên GPU

Notebook này dùng `train_full_grid_gpu.py` cùng thư mục (logic chung ở `reference/long_logic_engine.py`). Sửa **ô Cấu hình**, rồi chọn **Run All**. Kết quả lưu thành CSV cạnh notebook.

**Vào lệnh tại nến `i`:** close **quay lại trong dải dưới** (`close[i-1] < lower[i] < close[i]`) **và** `RSI(rsi_window)[i] < rsi_threshold` (quá bán).

**Stop (risk):** `sl_multiplier × (upper − lower)` (độ rộng dải, vì entry nằm sát dải dưới). TP = `tp_sl_ratio` × risk, timeout `MAX_HOLDING_BARS` nến, phí vào lệnh `0.001 / sl_ratio` (bỏ lệnh nếu > 0.2R), một vị thế mỗi lúc — giống bản RSI gốc.

**Phí giữ lệnh:** tính mỗi 8 giờ trên mọi khung (`FEE_BARS = None` ⇒ 32 ở 15min, 8 ở 1h). Bản long RSI gốc dùng 8 ở 15min (gấp 4 lần phía short); ở đây đã chuẩn hoá.

Không có giá trị tắt bộ lọc; ô so sánh nhóm theo `rsi_threshold`. Grid search trên toàn khoảng train là **in-sample**; đánh giá ngoài mẫu bằng `reference/walk_forward_15min.py`. Ô **Kiểm tra CPU** so GPU với replay float64; ô replay cuối dùng chính replay đó để liệt kê từng lệnh.

In [ ]:
# CẤU HÌNH — sửa ô này trước khi chạy
TIMEFRAME = "15min"                 # ví dụ "5min", "15min", "1h", "4h"
START_DATE = "2023-01-01"          # ngày bắt đầu, bao gồm ngày này
END_DATE = "2024-07-01"            # ngày kết thúc, bao gồm ngày này
DATA_CSV = None                     # None = CSV 5min mặc định của script
SCRIPT_PATH = None                  # None = tự tìm train_full_grid_gpu.py

SCORE_FORMULA = "r_martin_monthly"
# Sửa trực tiếp các danh sách dưới đây để đổi grid. Số bộ tham số là tích số giá trị mỗi danh sách.
SEARCH_SPACES = {
    "boll_window": [10, 14, 20, 25, 30, 40, 50],
    "boll_window_dev": [1.8, 2.0, 2.2, 2.4, 2.6, 2.8],
    "rsi_window": [7, 14, 21, 30],
    "rsi_threshold": [20, 25, 30, 35, 40, 45],
    "sl_multiplier": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    "tp_sl_ratio": [1.1],
}
LIMIT = None                         # None = toàn bộ bộ tham số; số nguyên để chạy thử
MAX_HOLDING_BARS = 288             # số nến tối đa giữ lệnh theo TIMEFRAME
FEE_BARS = None                    # None = 8 giờ mỗi lần phí (32 ở 15min)
VERIFY_SAMPLES = 0                  # 0 = bỏ qua so với CPU; ví dụ 8 = kiểm tra 8 bộ
MIN_TRADES = 50                     # bảng xếp hạng chỉ giữ bộ có nhiều lệnh hơn số này
OUTPUT_CSV = None                   # None = CSV theo timeframe, cạnh notebook

In [ ]:
# Nạp script cùng thư mục (cũng hoạt động khi kernel bắt đầu ở thư mục khác trong repo)
from pathlib import Path
import importlib.util
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SIDE_FOLDER = "long_bollinger+meanrev"
if SCRIPT_PATH is None:
    candidates = []
    for parent in (Path.cwd(), *Path.cwd().parents):
        if parent.name == SIDE_FOLDER:
            candidates.append(parent / "train_full_grid_gpu.py")
        candidates.append(parent / "reference" / SIDE_FOLDER / "train_full_grid_gpu.py")
    script_path = next((path.resolve() for path in candidates if path.is_file()), None)
    if script_path is None:
        raise FileNotFoundError("Không tìm thấy train_full_grid_gpu.py; hãy đặt SCRIPT_PATH ở ô Cấu hình")
else:
    script_path = Path(SCRIPT_PATH).expanduser().resolve()
if not script_path.is_file():
    raise FileNotFoundError(script_path)

spec = importlib.util.spec_from_file_location("avax_long_bollinger_meanrev", script_path)
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)
csv_path = Path(DATA_CSV).expanduser().resolve() if DATA_CSV else strategy.DEFAULT_CSV
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)
print("CSV:", csv_path)
print("Đã nạp:", script_path)
print("Phí giữ lệnh: mỗi", strategy.resolve_fee_bars(TIMEFRAME, FEE_BARS), "nến")

In [ ]:
# Đọc dữ liệu 5 phút, resample và tạo grid
strategy.timeframe_nanos(TIMEFRAME)
start_date, end_date = START_DATE, END_DATE
expected_keys = set(strategy.SEARCH_SPACE)
space = SEARCH_SPACES
if set(space) != expected_keys:
    raise ValueError(f"SEARCH_SPACES cần đúng các khóa: {sorted(expected_keys)}")
for name, values in space.items():
    if not isinstance(values, (list, tuple)) or not values:
        raise ValueError(f"{name} phải là danh sách có ít nhất một giá trị")
space = {name: list(values) for name, values in space.items()}

bars = strategy.load_bars(csv_path, TIMEFRAME, start_date, end_date)
combinations = itertools.product(*(space[name] for name in space))
if LIMIT is not None:
    combinations = itertools.islice(combinations, LIMIT)
grid = pd.DataFrame(combinations, columns=list(space))
print(f"Dữ liệu: {len(bars):,} nến; {bars['timestamp'].iloc[0]} → {bars['timestamp'].iloc[-1]}")
print(f"Grid: {len(grid):,} bộ tham số")
display(grid.head())

In [ ]:
# Chạy toàn bộ grid trên GPU
result, timing = strategy.run_gpu_grid(
    bars, grid, TIMEFRAME,
    max_holding_bars=MAX_HOLDING_BARS,
    fee_bars=FEE_BARS,
    score_formula=SCORE_FORMULA,
)
print("Thời gian:", timing)
print(f"Đã tính {len(result):,} bộ tham số")

In [ ]:
# TÍNH LẠI SCORE — chạy lại riêng ô này sau khi đổi SCORE_FORMULA
result = strategy.score_result(result, SCORE_FORMULA)
print("Score:", SCORE_FORMULA)

In [ ]:
# Kiểm tra CPU (tùy chọn): đối chiếu mẫu với replay float64
if VERIFY_SAMPLES:
    strategy.verify_samples(bars, result, VERIFY_SAMPLES, MAX_HOLDING_BARS, FEE_BARS, TIMEFRAME)

In [ ]:
# Lưu kết quả và xem các bộ tham số đứng đầu
output_path = (Path(OUTPUT_CSV).expanduser().resolve() if OUTPUT_CSV
               else script_path.with_name(f"full_gpu_grid_results_{TIMEFRAME}_fp32.csv"))
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False)

ranked = result.loc[(result["number_of_trade"] > MIN_TRADES) & result["score"].notna()].sort_values("score", ascending=False)

print("Đã lưu:", output_path)
print(f"Bộ có score hợp lệ và > {MIN_TRADES} lệnh: {len(ranked):,}/{len(result):,}")
display(ranked.head(20))

In [ ]:
# Bộ lọc có giúp không? Trung vị toàn grid theo từng giá trị của `rsi_threshold` (ít nhiễu hơn bộ "tốt nhất").
by_filter = result.groupby("rsi_threshold").agg(
    median_trades=("number_of_trade", "median"),
    median_performance=("performance", "median"),
    share_profitable=("performance", lambda s: (s > 0).mean()),
    median_r_per_month=("r_per_month", "median"),
    median_r_martin_monthly=("r_martin_monthly", "median"),
)
display(by_filter.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
labels = [("tắt" if value == 0 else f"{value:g}") for value in by_filter.index]
axes[0].bar(labels, by_filter["median_performance"])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(xlabel="rsi_threshold", ylabel="Performance trung vị (R)", title="Performance theo rsi_threshold")
axes[1].bar(labels, by_filter["share_profitable"])
axes[1].set(xlabel="rsi_threshold", ylabel="Tỷ lệ bộ có lãi", title="Tỷ lệ bộ tham số có lãi")
plt.tight_layout()
plt.show()

In [ ]:
# Biểu đồ để xem phân bố score và hiệu quả/rủi ro của grid
if ranked.empty:
    print("Không có score hợp lệ để vẽ")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    ranked["score"].plot.hist(ax=axes[0], bins=50, title="Phân bố score")
    axes[0].set_xlabel("Score")
    axes[1].scatter(ranked["max_dd"], ranked["performance"], s=5, alpha=0.3)
    axes[1].set(xlabel="max_dd", ylabel="Performance (R)", title="Performance và drawdown")
    plt.tight_layout()
    plt.show()

In [ ]:
# Điền ID của trial trong bảng `result` (đếm từ 0), rồi chạy riêng ô này. None = bộ đứng đầu `ranked`.
TRIAL_ID = None
BACKTEST_START_DATE = None  # Ví dụ "2024-07-01"; None = ngày bắt đầu train
BACKTEST_END_DATE = None    # Ví dụ "2025-06-20"; None = ngày kết thúc train

import numpy as np

if TRIAL_ID is None:
    if ranked.empty:
        raise ValueError("Bảng ranked rỗng; hãy điền TRIAL_ID")
    TRIAL_ID = int(ranked.index[0])
if TRIAL_ID not in result.index:
    raise KeyError(f"TRIAL_ID={TRIAL_ID} không có trong result.index")
params = grid.loc[TRIAL_ID].to_dict()
backtest_start = BACKTEST_START_DATE or start_date
backtest_end = BACKTEST_END_DATE or end_date
backtest_bars = (bars if BACKTEST_START_DATE is None and BACKTEST_END_DATE is None
                 else strategy.load_bars(csv_path, TIMEFRAME, backtest_start, backtest_end))

# GPU cho đúng một bộ tham số, và replay float64 liệt kê từng lệnh; hai bên phải khớp.
backtest_result, _ = strategy.run_gpu_grid(
    backtest_bars, grid.loc[[TRIAL_ID]].reset_index(drop=True), TIMEFRAME,
    max_holding_bars=MAX_HOLDING_BARS, fee_bars=FEE_BARS, score_formula=SCORE_FORMULA)
backtest_trial = backtest_result.iloc[0]
trades = strategy.cpu_trace(backtest_bars, params, TIMEFRAME, MAX_HOLDING_BARS, FEE_BARS)
replay = strategy.trade_metrics(backtest_bars, trades)
if replay["number_of_trade"] != int(backtest_trial["number_of_trade"]):
    raise AssertionError(f"Trial {TRIAL_ID}: GPU {int(backtest_trial['number_of_trade'])} lệnh, replay {replay['number_of_trade']}")
if not np.isclose(replay["performance"], float(backtest_trial["performance"]), rtol=1e-4, atol=1e-3):
    raise AssertionError(f"Trial {TRIAL_ID}: tổng R replay khác performance GPU")

print(f"Giai đoạn backtest: {backtest_bars['timestamp'].iloc[0]} đến {backtest_bars['timestamp'].iloc[-1]}")
display(backtest_trial.to_frame(name="backtest_trial").T)
print(f"Trial {TRIAL_ID}: {replay['number_of_trade']} lệnh; performance replay = {replay['performance']:.6f} R; "
      f"GPU = {backtest_trial['performance']:.6f} R")
display(trades["outcome"].value_counts().to_frame("số lệnh"))
display(trades.head(20))

if trades.empty:
    print("Không có lệnh nào trong khoảng backtest")
else:
    ordered = trades.sort_values("close_timestamp")
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.step(ordered["close_timestamp"], ordered["gross_r"].cumsum(), where="post",
            linestyle="--", label="Cumulative R (Before Fee)")
    ax.step(ordered["close_timestamp"], ordered["net_r"].cumsum(), where="post",
            color="green", label="Cumulative R (After Fee)")
    ax.set(xlabel="Time", ylabel="Cumulative R-Multiple",
           title=f"AVAX LONG meanrev — trial {TRIAL_ID} ({backtest_start} to {backtest_end})")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    plt.show()

In [ ]:
ax = backtest_bars.set_index("timestamp")["close"].plot(figsize=(18, 6))
ax.set(title="AVAX/USDT close", xlabel="Time", ylabel="Price (USDT)")
plt.show()